In [1]:
import os
import numpy as np
import rasterio as rio
from rasterio.merge import merge
from rasterio.plot import show
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
folder_path = "/home/jovyan/work/AVOCA/GEM/Production/Gapfilled/"  
output_folder = "/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/monthly_band2_zero/" 
files = sorted(Path(folder_path).glob("*.tiff"))
dates = [f.stem.split("_")[1] for f in files]
unique_months = sorted(set(date[:6] for date in dates))

In [ ]:
def reclassify(img):
    return np.where(img == 0, 1, 0).astype(np.uint32)

def process_monthly_averages(folder_path):

    for month in sorted(unique_months):    
        monthly_files = [f for f in files if f.stem.startswith(f"GEMLST_{month}")]
        with rio.open(monthly_files[0]) as img:
            first_img = img.read(2)
            count_img = reclassify(first_img)
            profile = img.profile
        
        # Open and read all files for the month
        for file in monthly_files[1:]:
            with rio.open(file) as src:
                data = src.read(2)
                data = reclassify(data)
                count_img += data

        # Save the monthly average as a GeoTIFF
        output_filename = f"MONTHLY_CLOUD_{month}.tiff"
        output_path = os.path.join(output_folder, output_filename)

        with rio.open(
            output_path,
            'w',
            driver='GTiff',
            height=count_img.shape[0],
            width=count_img.shape[1],
            count=1,
            dtype='uint32',
            crs=profile['crs'],
            transform=profile['transform']
        ) as dst:
            dst.write(count_img, 1)

        print(f"Saved: {output_path}")

# process_monthly_averages(folder_path)

In [6]:
path = "/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/monthly_band2_zero/" 
files = sorted(Path(path).glob("*.tiff"))
output_path = "/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/Validation_GEMLST/QA_band_zeros_total.tiff"

In [ ]:
with rio.open(files[0]) as src:
    first_img = src.read(1)
    shape = first_img.shape
    profile = src.profile

# Create an image to hold the total count of non-zero pixels across all files
total_count = np.zeros(shape, dtype=np.uint32)

for file in files:
    with rio.open(file) as src:
        data = src.read(1).astype(np.uint32)
        total_count += data

profile.update(dtype="uint32", count=1, nodata=0)

with rio.open(
    output_path,
    'w',
    driver='GTiff',
    height= shape[0],
    width= shape[1],
    count=1,
    dtype='uint32',
    crs=profile['crs'],
    transform=profile['transform']
) as dst:
    dst.write(total_count, 1)

print(f"Saved: {output_path}")


Saved: /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/Validation_GEMLST/QA_band_zeros_total.tiff


In [7]:
# How many valid pixels are there in a single GEMLST image?

with rio.open("/home/sirsimsius/Hentet/GEMLST_20251231.tiff") as src:
    data = src.read(2)
    valid_pixels = np.isfinite(data).sum()
    print(f"Valid pixels in GEMLST_20251231.tiff: {valid_pixels}")

with rio.open("/home/sirsimsius/Hentet/GEMLST_20000224.tiff") as src:
    data = src.read(2)
    valid_pixels = np.isfinite(data).sum()
    print(f"Valid pixels in GEMLST_20000224.tiff: {valid_pixels}")

Valid pixels in GEMLST_20251231.tiff: 2143385
Valid pixels in GEMLST_20000224.tiff: 2143385
